# Dự báo Giá Xăng Bán lẻ Việt Nam (Vietnam Retail Gasoline Price Forecasting)

**Tác giả**: Vũ Xuân Duy Anh — DSEB Year 2, Trường Đại học Kinh tế Quốc dân (NEU)  
**Mục tiêu**: Thực tập sinh Machine Learning — Viện Dầu khí Việt Nam (VPI)  

---

## 1. Giới thiệu & Lý do chọn bài toán

Tại Việt Nam, giá xăng dầu bán lẻ được điều hành bởi Liên Bộ Công Thương - Tài chính vào mỗi **Thứ Năm hàng tuần** (theo Nghị định 80/2023/NĐ-CP). Dự báo chính xác giá xăng trong kỳ điều hành tiếp theo mang lại giá trị thực tiễn rất lớn cho:
- **Doanh nghiệp đầu mối xăng dầu** (Petrolimex, PVOIL) trong việc tối ưu hóa tồn kho và nhập hàng.
- **Các cơ quan tham mưu chính sách** (như VPI, Bộ Kế hoạch và Đầu tư) trong việc dự báo chỉ số giá tiêu dùng CPI và lạm phát.

Dự án này tập trung vào **giá bán lẻ xăng nội địa Việt Nam (RON95 và E5 RON92)** — một bài toán đặc thù và có tính thực tế cao hơn nhiều so với bài toán dự báo giá dầu thô thế giới thông thường.

## 2. Thiết lập & Tải dữ liệu

Dữ liệu lịch sử giá bán lẻ xăng dầu được lấy từ bộ dữ liệu mã nguồn mở `toanqng/fuel`, bộ dữ liệu này ghi nguồn tham chiếu là Petrolimex. Project lưu snapshot đã làm sạch trong `data/` để tái lập kết quả. Các biến đặc trưng thị trường gồm Brent, WTI, USD Index (DXY), VIX và USD/VND được tải từ Yahoo Finance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
DATA_PATH = "data/merged_dataset.csv"

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f"Kích thước dữ liệu: {df.shape}")
df.head()

## 3. Phân tích Dữ liệu Khám phá (EDA)

Chúng ta sẽ vẽ biểu đồ biểu diễn xu hướng giá xăng bán lẻ Việt Nam so với biến động giá dầu thô Brent thế giới.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Thời gian')
ax1.set_ylabel('Giá xăng Việt Nam (VND/lít)', color=color)
ax1.plot(df['date'], df['RON95'], color=color, label='Xăng RON95', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Giá dầu Brent thế giới (USD/thùng)', color=color)
ax2.plot(df['date'], df['brent'], color=color, label='Dầu Brent', linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Mối liên hệ giữa giá xăng RON95 Việt Nam và giá dầu Brent thế giới (2019-2026)')
fig.tight_layout()
plt.show()

## 4. Xây dựng Biến Đặc trưng (Feature Engineering) & Tránh rò rỉ dữ liệu (Leakage)

### Quy tắc chống Leakage:
1. **Căn chỉnh dữ liệu thị trường**: Khi ghép dữ liệu, biến Brent/WTI/DXY/VIX/USDVND được dịch +1 ngày để tại ngày điều hành $t$ mô hình chỉ nhìn thấy giá thị trường đến ngày $t-1$.
2. **Lag của giá bán lẻ**: `RON95_lag1`, `RON95_lag2` dùng `.shift()` nên không bao giờ dùng giá kỳ hiện tại làm feature.
3. **Rolling MA/Std**: Tính trên chuỗi thị trường đã được căn chỉnh ở bước (1); mỗi hàng chỉ dùng quan sát cùng hàng và các hàng trước đó.
4. **Winsorizing**: Ngưỡng phân vị **chỉ fit trên train fold**, rồi mới transform test fold. Không winsorize target $Y$.
5. **Không back-fill**: Yahoo Finance chỉ `ffill()` để tránh đưa dữ liệu tương lai về quá khứ.

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.features import add_event_dummies, build_lag_and_rolling_features, get_feature_columns

# Thực hiện biến đổi đặc trưng
df_features = add_event_dummies(df)
df_features = build_lag_and_rolling_features(df_features)
feature_cols = get_feature_columns('RON95')
df_features = df_features.dropna(subset=feature_cols + ['RON95', 'E5_RON92']).reset_index(drop=True)
print(f"Số lượng features cho RON95: {len(feature_cols)}")

## 5. Đánh giá Mô hình với 6-Fold Walk-Forward Cross Validation

Mô hình được đánh giá trên 6 folds tương ứng với các năm từ 2021 đến 2026. Đây là phương pháp chuẩn mực cho dữ liệu chuỗi thời gian.

In [ ]:
summary_path = "output/model_comparison.csv"
if os.path.exists(summary_path):
    summary_df = pd.read_csv(summary_path)
    print(summary_df.to_string(index=False))
else:
    print("Hãy chạy file models.py hoặc pipeline.py để sinh kết quả so sánh mô hình!")

### Phân tích kết quả:
- Bảng trên được đọc trực tiếp từ `output/model_comparison.csv`, vì vậy luôn khớp với lần chạy pipeline mới nhất.
- **Naive Baseline** là benchmark bắt buộc vì giá điều hành thường có quán tính mạnh qua từng kỳ.
- **XGBoost** được dùng làm mô hình điểm khi là ML model có MAPE tốt nhất; **LightGBM Quantile** được dùng để tạo khoảng dự báo 2.5%-97.5%.

## 6. Đóng góp Nghiên cứu & Phân tích Độ nhạy

### 6.1 So sánh resampling chu kỳ 7 ngày vs 10 ngày
Kết quả định lượng được lưu trong `output/policy_cycle_comparison.csv` và biểu đồ `output/policy_analysis.png`. Pipeline tính weighted MAPE theo số quan sát từng fold để tránh làm lệch kết quả bởi các năm dữ liệu chưa hoàn chỉnh. Đây là stress test theo tần suất dữ liệu, không phải kết luận nhân quả về hiệu quả chính sách.

In [ ]:
from IPython.display import Image
policy_path = 'output/policy_cycle_comparison.csv'
if os.path.exists(policy_path):
    print(pd.read_csv(policy_path).to_string(index=False))
Image(filename='output/policy_analysis.png')

### 6.2 Bản đồ Độ nhạy (Sensitivity Heatmap)
Bản đồ này dự báo mức giá xăng RON95 dựa trên các biến động của giá dầu Brent thế giới và mức can thiệp của Quỹ Bình ổn giá (BOG).

In [ ]:
Image(filename='output/sensitivity_heatmap.png')

## 7. Dự báo Giá xăng cho Kỳ Điều hành Tiếp theo

Dưới đây là giá dự báo cho kỳ điều hành Thứ Năm tuần tới. Điểm dự báo dùng **XGBoost**; khoảng dự báo 2.5%-97.5% dùng **LightGBM Quantile Regression**; cột Naive Baseline được giữ để đối chiếu với giá kỳ gần nhất.

In [ ]:
pred_path = "output/forecast_this_thursday.csv"
if os.path.exists(pred_path):
    pred_df = pd.read_csv(pred_path)
    print(pred_df.to_string(index=False))
else:
    print("Hãy chạy file analysis.py hoặc pipeline.py để tạo dự báo!")